# Ladder-operator expansion of $\sigma^x_{\rm eff}$ for GKP state preparation

This notebook takes Eq. (19) of *GKP Brachistochrone* (Aidan's notes, §1.2) and expands the
effective operator

$$\sigma^x_{\rm eff} \;=\; |\psi_i\rangle\langle\psi_f^\perp| + |\psi_f^\perp\rangle\langle\psi_i|,
\qquad |\psi_i\rangle=|0\rangle,\quad |\psi_f\rangle = |\tilde\mu_L\rangle$$

into a normal-ordered series $\sum_{n,k} c_{n,k}\,\hat a^{\dagger\,n+k}\hat a^{k} + {\rm h.c.}$, ordered by
increasing powers of the ladder operators.

**Conventions fixed for this notebook** (chosen in discussion, and they differ from the notes in one place):

| item | choice |
|---|---|
| $\lvert\psi_f^\perp\rangle$ | **normalized**, i.e. $\bigl(\lvert\tilde\mu_L\rangle - \langle 0\lvert\tilde\mu_L\rangle\lvert 0\rangle\bigr)/\sqrt{1-\lvert s\rvert^2}$, matching `brachistochrone.py` (`.unit()`), *not* the unnormalized Eq. (13) |
| truncation | **independent caps** $1\le n\le N_{\max}$ and $0\le k\le K_{\max}$ |
| coefficients | left as **unevaluated lattice sums** $S_n^{(\mu)}$ (symbols; the `Sum` objects are displayed but never evaluated) |
| ordering | normal-ordered, $\hat a^{\dagger}$ to the left |

Sections:

1. Setup, SymBosonKit import (with a local fallback shim)
2. The envelope push-through identity $e^{-\Delta^2\hat n}e^{\zeta\hat a^\dagger}|0\rangle = e^{\zeta^\Delta \hat a^\dagger}|0\rangle$, verified with the library
3. Lattice moments $S_n^{(\mu)}$, and why they are the *only* state-dependent data in the problem
4. Normal-ordered vacuum projector and its truncation
5. Assembly of $\sigma^{x}_{\rm eff}$ (and $\sigma^{y}_{\rm eff}$) order by order
6. Term table, Hermiticity check, grading by total ladder degree and circuit-QED reading
7. Exact truncation-error formula, verified symbolically and numerically
8. Optional numerics: evaluating $S_n$ on a square lattice, and the exact convention map to `states.py`

## 1. Setup

SymBosonKit is installed by copying `symbosonkit.py` next to the notebook (no package on PyPI), so the
import is guarded. Everything used from it — `similarity_transform` / `_ad` for the push-through identity
and `collect_terms` / `coeff_of` for coefficient extraction on non-commuting products — has a small local
fallback so the notebook runs either way. When the library *is* present its answers are cross-checked
against the fallback rather than trusted blindly.

In [1]:
import itertools
from math import comb, factorial, sqrt, pi

import numpy as np
import sympy as sp
from sympy.physics.quantum import Commutator, Dagger
from sympy.physics.quantum.boson import BosonOp
from sympy.physics.quantum.operatorordering import normal_ordered_form
from IPython.display import Math, display

sp.init_printing(use_latex="mathjax")

try:
    from gkp_optimal_control.symbosonkit import _ad, coeff_of, collect_terms, similarity_transform
    HAVE_SYMBOSONKIT = True
except ModuleNotFoundError:
    HAVE_SYMBOSONKIT = False

print("sympy", sp.__version__, "| SymBosonKit:", "found" if HAVE_SYMBOSONKIT else "not found (using fallbacks)")

a = BosonOp("a")
adag = Dagger(a)
Delta, omega = sp.symbols("Delta omega", positive=True)
alpha, beta = sp.symbols("alpha beta")
j, l = sp.symbols("j l", integer=True)
n_sym, k_sym, q_sym, K_sym = sp.symbols("n k q K", integer=True, nonnegative=True)

# S[n] == S_n^{(mu)}: the n-th lattice moment, left unevaluated throughout.
S = sp.IndexedBase("S")
Nrm = sp.Symbol("N^Delta_mu", positive=True)   # state normalization
R = sp.Symbol("R", positive=True)              # 1/sqrt(1 - |s|^2) from normalizing |psi_f^perp>

sympy 1.14.0 | SymBosonKit: found


### Coefficient extraction on non-commuting products

`sympy.Expr.coeff` is unreliable once `BosonOp`s are involved, which is exactly the gap SymBosonKit's
`collect_terms` / `coeff_of` fill. The local fallback splits every term with `args_cnc()` into its
commutative part and its non-commutative word, then matches words. It is exact for the normal-ordered
monomial basis used here.

In [2]:
def monomial_word(mono):
    """Non-commutative word (list of BosonOp powers) of a monomial."""
    _, nc = sp.Mul(mono).args_cnc(split_1=False)
    return nc


def local_coeff_of(expr, mono):
    """Scalar coefficient of `mono` in `expr`, matching non-commutative words."""
    target = monomial_word(mono)
    total = sp.S.Zero
    for term in sp.Add.make_args(sp.expand(expr)):
        c, nc = term.args_cnc(split_1=False)
        if nc == target:
            total += sp.Mul(*c)
    return sp.simplify(total)


def monomial_coeff(expr, mono, cross_check=None):
    """Coefficient of `mono`; cross-checks SymBosonKit's `coeff_of` when available."""
    value = local_coeff_of(expr, mono)
    if cross_check is None:
        cross_check = HAVE_SYMBOSONKIT
    if cross_check:
        try:
            other = coeff_of(collect_terms(sp.expand(expr), mono), mono)
        except Exception as exc:  # signature/behaviour drift in the library
            print(f"  [note] SymBosonKit coeff_of unavailable for {mono}: {exc!r}")
        else:
            if sp.simplify(other - value) != 0:
                print(f"  [warn] coeff_of mismatch for {mono}: {other} vs {value}")
    return value


# quick self-test
_test = S[1] * adag + sp.conjugate(S[1]) * a - 3 * S[2] * adag**2 * a
assert local_coeff_of(_test, adag) == S[1]
assert local_coeff_of(_test, adag**2 * a) == -3 * S[2]
assert local_coeff_of(_test, adag * a) == 0
print("coefficient extractor OK")

coefficient extractor OK


## 2. The envelope push-through identity

The finite-energy code states carry the Gaussian envelope $e^{-\Delta^2\hat a^\dagger\hat a}$, and the whole
derivation hinges on Eq. (14)–(15) of the notes,

$$e^{-\Delta^2\hat n}\,e^{\zeta \hat a^\dagger}\,e^{-(-\Delta^2\hat n)} = e^{\zeta^\Delta \hat a^\dagger},
\qquad \zeta^\Delta \equiv e^{-\Delta^2}\zeta ,$$

which follows from $\mathrm{ad}^n_{A}(\hat a^\dagger) = (-\Delta^2)^n \hat a^\dagger$ with $A=-\Delta^2\hat n$.
That nested-commutator ladder is what `_ad` / `similarity_transform` compute, so we verify it directly
instead of taking it on faith. Note the series closes in closed form (geometric in $-\Delta^2$), so no
BCH truncation error enters anywhere below.

In [3]:
A_env = -Delta**2 * adag * a


def ad_power(A, X, order):
    """ad_A^order(X) = [A,[A,...,[A,X]]]; local stand-in for SymBosonKit's _ad."""
    out = sp.expand(X)
    for _ in range(order):
        out = sp.expand(Commutator(A, out).doit())
    return normal_ordered_form(out)


rows = []
for order in range(5):
    local = sp.simplify(ad_power(A_env, adag, order))
    expected = (-Delta**2) ** order * adag
    assert sp.simplify(local - expected) == 0
    if HAVE_SYMBOSONKIT:
        try:
            lib = sp.simplify(normal_ordered_form(sp.expand(_ad(A_env, adag, order))))
            assert sp.simplify(lib - expected) == 0, (order, lib)
        except TypeError as exc:
            print(f"  [note] _ad signature differs ({exc!r}); relying on local ad_power")
    rows.append((order, expected))

display(Math(r"\mathrm{ad}^n_{A}(\hat a^\dagger)\ \text{for}\ A=-\Delta^2\hat n:\quad "
             + ",\\quad ".join(sp.latex(e) for _, e in rows)))
display(Math(r"\sum_{n\ge 0}\frac{(-\Delta^2)^n}{n!}\,\hat a^\dagger = "
             + sp.latex(sp.exp(-Delta**2) * adag)
             + r"\qquad\Longrightarrow\qquad \zeta^\Delta = e^{-\Delta^2}\zeta"))

  [note] _ad signature differs (TypeError('_ad() takes 2 positional arguments but 3 were given')); relying on local ad_power
  [note] _ad signature differs (TypeError('_ad() takes 2 positional arguments but 3 were given')); relying on local ad_power
  [note] _ad signature differs (TypeError('_ad() takes 2 positional arguments but 3 were given')); relying on local ad_power
  [note] _ad signature differs (TypeError('_ad() takes 2 positional arguments but 3 were given')); relying on local ad_power
  [note] _ad signature differs (TypeError('_ad() takes 2 positional arguments but 3 were given')); relying on local ad_power


<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 3. The lattice moments $S_n^{(\mu)}$

Combining Eq. (9) with the push-through identity, the approximate code state is a superposition of
*unnormalized* exponentials of $\hat a^\dagger$:

$$|\tilde\mu_L\rangle = \mathcal N^\Delta_\mu \sum_{\substack{m,l\in\mathbb Z\\ m\equiv\mu\,(2)}}
\underbrace{e^{-i\pi m l/2}\,e^{-|\zeta_{m,l}|^2/2}}_{\textstyle c_{m,l}}\; e^{\zeta^\Delta_{m,l}\hat a^\dagger}|0\rangle,
\qquad \zeta_{m,l}=m\alpha+l\beta .$$

Writing $m = 2j+\mu$ and expanding $e^{\zeta^\Delta\hat a^\dagger}|0\rangle=\sum_p (\zeta^\Delta)^p|p\rangle/\sqrt{p!}$,
*every* quantity below is a function of the single family of lattice sums

$$\boxed{\;S^{(\mu)}_n \;\equiv\; \sum_{j,l\in\mathbb Z} e^{-i\pi(2j+\mu)l/2}\,
e^{-|\zeta_{2j+\mu,l}|^2/2}\,\bigl(e^{-\Delta^2}\zeta_{2j+\mu,l}\bigr)^{n}\;}$$

because $\langle p|\tilde\mu_L\rangle = \mathcal N^\Delta_\mu\,S^{(\mu)}_p/\sqrt{p!}$. In particular

$$\mathcal N^\Delta_\mu = \Bigl(\sum_{p\ge 0}\frac{|S_p|^2}{p!}\Bigr)^{-1/2},\qquad
s \equiv \langle 0|\tilde\mu_L\rangle = \mathcal N^\Delta_\mu S_0,\qquad
R \equiv \frac{1}{\sqrt{1-|s|^2}} .$$

So the $S_n$ are not just bookkeeping: they are the Fock amplitudes of the target state, up to $\sqrt{n!}$.
They stay symbolic from here on.

In [4]:
def zeta(jj, ll, mu):
    return (2 * jj + mu) * alpha + ll * beta


def moment_sum(n, mu, jj=j, ll=l):
    """Unevaluated lattice sum defining S_n^{(mu)} (display only)."""
    z = zeta(jj, ll, mu)
    term = (sp.exp(-sp.I * sp.pi * (2 * jj + mu) * ll / 2)
            * sp.exp(-sp.Abs(z) ** 2 / 2)
            * (sp.exp(-Delta**2) * z) ** n)
    return sp.Sum(term, (jj, -sp.oo, sp.oo), (ll, -sp.oo, sp.oo))


for mu in (0, 1):
    display(Math(rf"S^{{({mu})}}_n = " + sp.latex(moment_sum(n_sym, mu))))

display(Math(r"S^{(0)}_1 = " + sp.latex(moment_sum(1, 0))))
display(Math(r"\mathcal N^\Delta_\mu = " + sp.latex(
    1 / sp.sqrt(sp.Sum(sp.Abs(S[n_sym]) ** 2 / sp.factorial(n_sym), (n_sym, 0, sp.oo))))
    + r",\qquad s = \mathcal N^\Delta_\mu S_0,\qquad R = (1-|s|^2)^{-1/2}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4. Normal-ordered vacuum projector

The other ingredient is the vacuum projector in normal-ordered form,

$$|0\rangle\langle 0| \;=\; \,:e^{-\hat a^\dagger \hat a}:\; = \sum_{k=0}^{\infty}\frac{(-1)^k}{k!}\hat a^{\dagger k}\hat a^{k},$$

and $k$ is the second truncation index. The partial sum is *not* a projector, but its action on Fock
states is known in closed form:

$$P_K|q\rangle \equiv \sum_{k=0}^{K}\frac{(-1)^k}{k!}\hat a^{\dagger k}\hat a^{k}|q\rangle
= \Bigl[\textstyle\sum_{k=0}^{K}(-1)^k\binom{q}{k}\Bigr]|q\rangle
= (-1)^K\binom{q-1}{K}\,|q\rangle .$$

Since $\binom{q-1}{K}=0$ for $1\le q\le K$, **$P_K$ is exactly $|0\rangle\langle0|$ on the subspace
$q\le K$** and leaks only on columns $q>K$. That is the precise meaning of the $k$-truncation, and it is
verified symbolically below.

In [5]:
def vacuum_projector(Kmax):
    """Truncated normal-ordered vacuum projector sum_{k<=Kmax} (-1)^k/k! adag^k a^k."""
    return sp.Add(*[sp.Integer(-1) ** k / sp.factorial(k) * adag**k * a**k
                    for k in range(Kmax + 1)])


for Kmax in range(3):
    display(Math(rf"P_{{{Kmax}}} = " + sp.latex(vacuum_projector(Kmax))))

# symbolic identity: sum_{k=0}^{K} (-1)^k binom(q,k) = (-1)^K binom(q-1,K)
lhs = sp.Sum(sp.Integer(-1) ** k_sym * sp.binomial(q_sym, k_sym), (k_sym, 0, K_sym))
rhs = sp.Integer(-1) ** K_sym * sp.binomial(q_sym - 1, K_sym)
bad = [(qv, Kv) for qv in range(9) for Kv in range(9)
       if sp.simplify(lhs.subs({q_sym: qv, K_sym: Kv}).doit()
                      - rhs.subs({q_sym: qv, K_sym: Kv})) != 0]
print("projector-truncation identity violations for q,K <= 8:", bad)
display(Math(sp.latex(sp.Eq(lhs, rhs, evaluate=False))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

projector-truncation identity violations for q,K <= 8: []


<IPython.core.display.Math object>

## 5. Assembling $\sigma^{x}_{\rm eff}$

With $|\psi_i\rangle=|0\rangle$ and the normalized orthogonal component,

$$|\psi_f^\perp\rangle = R\,\mathcal N^\Delta_\mu \sum_{m,l} c_{m,l}\bigl(e^{\zeta^\Delta\hat a^\dagger}-1\bigr)|0\rangle
\;\Longrightarrow\;
|\psi_f^\perp\rangle\langle 0| = R\,\mathcal N^\Delta_\mu \sum_{n\ge1}\frac{S_n}{n!}\hat a^{\dagger n}\;
\sum_{k\ge0}\frac{(-1)^k}{k!}\hat a^{\dagger k}\hat a^{k},$$

so that

$$\boxed{\;\sigma^{x}_{\rm eff} = R\,\mathcal N^\Delta_\mu
\sum_{n=1}^{N_{\max}}\sum_{k=0}^{K_{\max}} \frac{(-1)^k}{n!\,k!}
\Bigl[S_n\,\hat a^{\dagger\,n+k}\hat a^{k} \;+\; S_n^{*}\,\hat a^{\dagger k}\hat a^{\,k+n}\Bigr]\;}$$

Two remarks:

* the "$-1$" in $(e^{\zeta^\Delta \hat a^\dagger}-1)$ is exactly why the sum starts at $n=1$: the $n=0$ piece is
  the projection along $|0\rangle$ that was subtracted off;
* relative to Eq. (19) the only change from normalizing $|\psi_f^\perp\rangle$ is the overall scalar $R$ —
  the operator content is untouched. With $\|H\|$ fixed this rescaling matters, since Eq. (19)'s
  unnormalized $\sigma^x$ has spectral norm $\sqrt{1-|s|^2}$ rather than 1.

$\sigma^{y}_{\rm eff} = -i\bigl(|\psi_i\rangle\langle\psi_f^\perp| - |\psi_f^\perp\rangle\langle\psi_i|\bigr)$
comes from the same two pieces and is built by the same function.

In [6]:
def creation_series(Nmax, coeff=S):
    """sum_{n=1}^{Nmax} S_n/n! adag^n  -- the (e^{zeta^Delta adag} - 1) factor."""
    return sp.Add(*[coeff[n] / sp.factorial(n) * adag**n for n in range(1, Nmax + 1)])


def outer_products(Nmax, Kmax):
    """(|psi_f^perp><0|, |0><psi_f^perp|) as normal-ordered polynomials, prefactors stripped."""
    lower = sp.expand(creation_series(Nmax) * vacuum_projector(Kmax))  # adag^{n+k} a^k
    upper = sp.expand(sp.Add(*[sp.conjugate(monomial_coeff(lower, adag ** (n + k) * a**k, False))
                               * adag**k * a ** (n + k)
                               for n in range(1, Nmax + 1) for k in range(Kmax + 1)]))
    return lower, upper


def sigma_eff(Nmax, Kmax, which="x", prefactor=True):
    """Truncated sigma^{x,y}_eff in normal-ordered ladder form."""
    lower, upper = outer_products(Nmax, Kmax)
    op = (upper + lower) if which == "x" else -sp.I * (upper - lower)
    if prefactor:
        op = sp.expand(R * Nrm * op)
    return op


for (Nmax, Kmax) in [(1, 0), (2, 1)]:
    display(Math(rf"\sigma^x_{{\rm eff}}\big|_{{N_{{\max}}={Nmax},\,K_{{\max}}={Kmax}}} = "
                 + sp.latex(sigma_eff(Nmax, Kmax))))

display(Math(r"\sigma^y_{\rm eff}\big|_{N_{\max}=1,\,K_{\max}=1} = "
             + sp.latex(sigma_eff(1, 1, which="y"))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Already normal-ordered, and Hermitian

Two structural checks: the construction should need no reordering (`normal_ordered_form` is a fixed point),
and $\sigma^x_{\rm eff}$ should be Hermitian term by term. Note also that every monomial has strictly
unequal creation/annihilation degree ($p-q=n\ge1$ or $\le-1$), so $\sigma^x_{\rm eff}$ has **no**
photon-number-diagonal part — no $\hat a^\dagger\hat a$, no Kerr $\hat a^{\dagger2}\hat a^2$, nothing a SNAP
gate alone could generate.

In [7]:
for (Nmax, Kmax) in [(2, 2), (3, 2)]:
    op = sigma_eff(Nmax, Kmax)
    assert sp.expand(normal_ordered_form(op) - op) == 0
    herm = sp.expand(op - Dagger(op))
    herm = sp.simplify(herm) if herm != 0 else herm
    print(f"(Nmax,Kmax)=({Nmax},{Kmax}): normal-ordered OK, op - op^dag = {herm}")

    opy = sigma_eff(Nmax, Kmax, which="y")
    hermy = sp.expand(opy - Dagger(opy))
    hermy = sp.simplify(hermy) if hermy != 0 else hermy
    print(f"                    sigma^y Hermitian: {hermy == 0}")

(Nmax,Kmax)=(2,2): normal-ordered OK, op - op^dag = 0
                    sigma^y Hermitian: True
(Nmax,Kmax)=(3,2): normal-ordered OK, op - op^dag = 0
                    sigma^y Hermitian: True


## 6. Term table and grading by ladder degree

Each $(n,k)$ contributes the single normal-ordered monomial pair $\hat a^{\dagger\,p}\hat a^{q}$ with

$$p = n+k,\qquad q = k,\qquad \text{i.e.}\quad n = p-q,\ k=q .$$

So the double sum is nothing but a relabelling of matrix-element bands: $n$ is the photon-number *change*
and $k$ is the column index. Grading by the total ladder degree $d = p+q = n+2k$ gives the usual
circuit-QED reading of a $d$-wave-mixing process.

In [8]:
def term_table(Nmax, Kmax, which="x"):
    op = sigma_eff(Nmax, Kmax, which=which, prefactor=False)
    rows = []
    for n in range(1, Nmax + 1):
        for k in range(Kmax + 1):
            mono = adag ** (n + k) * a**k
            rows.append(dict(n=n, k=k, p=n + k, q=k, degree=n + 2 * k,
                             mono=mono, coeff=monomial_coeff(op, mono)))
    return sorted(rows, key=lambda r: (r["degree"], r["p"]))


PROCESS = {1: "single-photon drive (displacement)",
           2: "two-wave mixing (squeezing)",
           3: "three-wave mixing (SNAIL / g3-type)",
           4: "four-wave mixing (Kerr-type)"}

rows = term_table(3, 2)
lines = ["| $d=p+q$ | $(n,k)$ | monomial | coefficient $\\times R\\,\\mathcal N^\\Delta_\\mu$ | process |",
         "|---|---|---|---|---|"]
for r in rows:
    lines.append("| {d} | ({n},{k}) | ${mono}$ | ${c}$ | {proc} |".format(
        d=r["degree"], n=r["n"], k=r["k"], mono=sp.latex(r["mono"]), c=sp.latex(r["coeff"]),
        proc=PROCESS.get(r["degree"], f"{r['degree']}-wave mixing")))
table = "\n".join(lines)
try:
    from IPython.display import Markdown
    display(Markdown(table))
except ImportError:
    print(table)

| $d=p+q$ | $(n,k)$ | monomial | coefficient $\times R\,\mathcal N^\Delta_\mu$ | process |
|---|---|---|---|---|
| 1 | (1,0) | ${{a}^\dagger}$ | ${S}_{1}$ | single-photon drive (displacement) |
| 2 | (2,0) | ${{a}^\dagger}^{2}$ | $\frac{{S}_{2}}{2}$ | two-wave mixing (squeezing) |
| 3 | (1,1) | ${{a}^\dagger}^{2} {a}$ | $- {S}_{1}$ | three-wave mixing (SNAIL / g3-type) |
| 3 | (3,0) | ${{a}^\dagger}^{3}$ | $\frac{{S}_{3}}{6}$ | three-wave mixing (SNAIL / g3-type) |
| 4 | (2,1) | ${{a}^\dagger}^{3} {a}$ | $- \frac{{S}_{2}}{2}$ | four-wave mixing (Kerr-type) |
| 5 | (1,2) | ${{a}^\dagger}^{3} {a}^{2}$ | $\frac{{S}_{1}}{2}$ | 5-wave mixing |
| 5 | (3,1) | ${{a}^\dagger}^{4} {a}$ | $- \frac{{S}_{3}}{6}$ | 5-wave mixing |
| 6 | (2,2) | ${{a}^\dagger}^{4} {a}^{2}$ | $\frac{{S}_{2}}{4}$ | 6-wave mixing |
| 7 | (3,2) | ${{a}^\dagger}^{5} {a}^{2}$ | $\frac{{S}_{3}}{12}$ | 7-wave mixing |

In [9]:
def graded_blocks(Nmax, Kmax, which="x"):
    """{total degree d: sum of all terms of that degree}, h.c. partners included."""
    op = sigma_eff(Nmax, Kmax, which=which, prefactor=False)
    blocks = {}
    for n in range(1, Nmax + 1):
        for k in range(Kmax + 1):
            d = n + 2 * k
            m_lo, m_hi = adag ** (n + k) * a**k, adag**k * a ** (n + k)
            piece = (monomial_coeff(op, m_lo, False) * m_lo
                     + monomial_coeff(op, m_hi, False) * m_hi)
            blocks[d] = sp.expand(blocks.get(d, 0) + piece)
    return dict(sorted(blocks.items()))


for d, block in graded_blocks(4, 2).items():
    display(Math(rf"d={d}:\quad " + sp.latex(sp.factor_terms(block))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

The $d=1$ block, $\;R\mathcal N^\Delta_\mu\,(S_1\hat a^\dagger + S_1^*\hat a)$, is a pure displacement
generator: the leading piece of the time-optimal Hamiltonian is a drive with amplitude set by the first
lattice moment. Higher $d$ add squeezing, then three- and four-wave mixing with coefficients that fall off
like $S_n/(n!k!)$ — which is what makes the ECD/SNAP decompositions in `gate_optimization.py` a sensible
target: the exact generator is an infinite-order non-Gaussian polynomial, but a *graded* one.

## 7. Exact truncation error

Because $p=n+k$ and $q=k$, the matrix elements of the truncated operator can be written down in closed
form. For $p>q$ (the upper band; the lower band is the conjugate):

$$\langle p|\sigma^x_{\rm eff}|q\rangle\Big|_{N_{\max},K_{\max}}
= R\mathcal N^\Delta_\mu\, S_{p-q}\frac{\sqrt{p!\,q!}}{(p-q)!\,q!}\sum_{k=0}^{\min(K_{\max},q)}(-1)^k\binom{q}{k},$$

and with the identity from §4 the error relative to the exact rank-one operator is

$$\langle p|\sigma^x_{\rm trunc}-\sigma^x_{\rm exact}|q\rangle =
\begin{cases}
-\,R\mathcal N^\Delta_\mu\,S_{p}/\sqrt{p!}, & q=0,\ p>N_{\max},\\[4pt]
R\mathcal N^\Delta_\mu\,S_{p-q}\dfrac{\sqrt{p!q!}}{(p-q)!\,q!}(-1)^{K_{\max}}\binom{q-1}{K_{\max}}, & 1\le p-q\le N_{\max},\ q>K_{\max},\\[4pt]
0 & \text{otherwise.}
\end{cases}$$

Two practical consequences:

* $N_{\max}$ controls how far up the Fock ladder the *physical* column $q=0$ is reproduced — it must cover
  the whole Fock support of $|\tilde\mu_L\rangle$, which for GKP envelopes is a slowly decaying tail (see §8);
* $K_{\max}$ only ever removes *spurious* matrix elements at $q>K_{\max}$. Those columns are never populated
  by the brachistochrone dynamics starting from vacuum, but they do change $\|H\|$ and hence the
  Mandelstam–Tamm time bound, so they matter for any statement about optimal duration.

Verified below against a direct Fock-basis construction with random moments.

In [10]:
def fock_ops(dim):
    a_mat = np.diag(np.sqrt(np.arange(1, dim)), 1)
    return a_mat, a_mat.conj().T


def sigma_x_trunc_matrix(S_vals, Nmax, Kmax, dim):
    a_mat, ad_mat = fock_ops(dim)
    M = np.zeros((dim, dim), complex)
    for n in range(1, Nmax + 1):
        for k in range(Kmax + 1):
            mono = np.linalg.matrix_power(ad_mat, n + k) @ np.linalg.matrix_power(a_mat, k)
            M += (-1) ** k / (factorial(n) * factorial(k)) * (S_vals[n] * mono
                                                              + np.conj(S_vals[n]) * mono.conj().T)
    return M


def sigma_x_exact_matrix(S_vals, dim):
    """R*N stripped: |psi_perp><0| + h.c. with <p|psi_perp> = S_p/sqrt(p!), p >= 1."""
    v = np.array([0.0 + 0j] + [S_vals[p] / sqrt(factorial(p)) for p in range(1, dim)])
    e0 = np.zeros(dim); e0[0] = 1.0
    return np.outer(v, e0) + np.outer(e0, v.conj())


def predicted_error(S_vals, Nmax, Kmax, dim):
    E = np.zeros((dim, dim), complex)
    for p in range(dim):
        for q in range(p):
            n = p - q
            if n > Nmax:
                if q == 0:
                    E[p, q] = -S_vals[p] / sqrt(factorial(p))
            elif q > Kmax:
                E[p, q] = (S_vals[n] * sqrt(factorial(p) * factorial(q))
                           / (factorial(n) * factorial(q)) * (-1) ** Kmax * comb(q - 1, Kmax))
    return E + E.conj().T


rng = np.random.default_rng(7)
dim = 9
S_rand = {n: rng.normal() + 1j * rng.normal() for n in range(0, 4 * dim)}
exact = sigma_x_exact_matrix(S_rand, dim)

print(f"{'Nmax':>5}{'Kmax':>6}   max|trunc-exact|   max|trunc-exact-formula|")
for Nmax, Kmax in [(1, 0), (2, 1), (3, 2), (5, 4), (dim - 1, dim - 1)]:
    T = sigma_x_trunc_matrix(S_rand, Nmax, Kmax, dim)
    print(f"{Nmax:>5}{Kmax:>6}   {np.abs(T - exact).max():16.3e}   "
          f"{np.abs(T - exact - predicted_error(S_rand, Nmax, Kmax, dim)).max():.3e}")
print("\n=> exact once Nmax, Kmax >= dim-1; error matches the closed form everywhere else.")

 Nmax  Kmax   max|trunc-exact|   max|trunc-exact-formula|
    1     0          2.636e+00   4.441e-16
    2     1          2.041e+01   5.403e-15
    3     2          4.082e+01   2.247e-14
    5     4          3.953e+01   1.785e-14
    8     8          1.157e-14   1.157e-14

=> exact once Nmax, Kmax >= dim-1; error matches the closed form everywhere else.


## 8. Optional: evaluating the moments, and the map to `states.py`

The coefficients above stay symbolic by design, but it is worth pinning down *how* they would be evaluated
and how the notes' state convention relates to the project code. For a square GKP lattice with
$q=(\hat a+\hat a^\dagger)/\sqrt2$, the choice $\alpha=\sqrt{\pi/2},\ \beta=i\sqrt{\pi/2}$ puts the logical-$Z$
peaks at multiples of $2\sqrt\pi$ in $q$.

**Convention map (worth knowing before comparing against the project code).** `states.py::gkp_states`
builds peaks at $\zeta_{m,l}$ with weight $\exp[-\tfrac12(1-e^{-2\Delta^2})|\zeta|^2]$, whereas the notes
apply $e^{-\Delta^2\hat n}$ *after* displacing, which both rescales the peak positions by $e^{-\Delta^2}$ and
weights them by $|\zeta|^2$ *before* rescaling. The two constructions are identical iff

$$\alpha \to \alpha e^{-\Delta^2},\quad \beta \to \beta e^{-\Delta^2},\qquad
\Delta \to \Delta_{\rm eff} = \sqrt{-\tfrac12\ln\!\bigl(2-e^{2\Delta^2}\bigr)}\;,$$

which requires $\Delta<\sqrt{\ln 2/2}\approx0.589$. Verified to machine precision below. Without that map
the two differ by a few $10^{-3}$ in overlap at $\Delta=0.3$ — small enough to look like a lattice-cutoff
artifact, which is exactly the kind of convention slip worth catching early.

In [11]:
def coherent_vector(dim, z):
    if z == 0:
        v = np.zeros(dim, complex); v[0] = 1.0; return v
    p = np.arange(dim)
    log_fact = np.concatenate(([0.0], np.cumsum(np.log(np.arange(1, dim)))))
    log_mag = -abs(z) ** 2 / 2 + p * np.log(abs(z)) - 0.5 * log_fact
    return np.exp(log_mag) * np.exp(1j * p * np.angle(z))


def lattice_moments(nmax, mu, alpha_v, beta_v, delta_v, cutoff):
    """S_n^{(mu)} for n = 0..nmax by direct lattice summation (notes' convention)."""
    S_vals = np.zeros(nmax + 1, complex)
    for jj in range(-cutoff, cutoff + 1):
        for ll in range(-cutoff, cutoff + 1):
            m = 2 * jj + mu
            z = m * alpha_v + ll * beta_v
            c = np.exp(-1j * pi * m * ll / 2) * np.exp(-abs(z) ** 2 / 2)
            zd = np.exp(-delta_v**2) * z
            S_vals += c * zd ** np.arange(nmax + 1)
    return S_vals


def state_from_moments(dim, mu, alpha_v, beta_v, delta_v, cutoff):
    S_vals = lattice_moments(dim - 1, mu, alpha_v, beta_v, delta_v, cutoff)
    log_fact = np.concatenate(([0.0], np.cumsum(np.log(np.arange(1, dim)))))
    v = S_vals / np.exp(0.5 * log_fact)
    return v / np.linalg.norm(v), S_vals


def state_statespy_convention(dim, mu, alpha_v, beta_v, delta_v, cutoff):
    """NumPy mirror of states.py::gkp_states (JAX is unavailable in this kernel path)."""
    env = 0.5 * (1.0 - np.exp(-2 * delta_v**2))
    tot = np.zeros(dim, complex)
    for jj in range(-cutoff, cutoff + 1):
        for ll in range(-cutoff, cutoff + 1):
            z = (2 * jj + mu) * alpha_v + ll * beta_v
            tot += (np.exp(-1j * pi * (jj + mu / 2) * ll) * np.exp(-env * abs(z) ** 2)
                    * coherent_vector(dim, z))
    return tot / np.linalg.norm(tot)


def delta_eff(delta_v):
    return np.sqrt(-0.5 * np.log(2 - np.exp(2 * delta_v**2)))


DIM, CUTOFF = 60, 4
ALPHA, BETA, DELTA = sqrt(pi / 2), 1j * sqrt(pi / 2), 0.30

for mu in (0, 1):
    psi, S_vals = state_from_moments(DIM, mu, ALPHA, BETA, DELTA, CUTOFF)
    mapped = state_statespy_convention(DIM, mu, ALPHA * np.exp(-DELTA**2),
                                       BETA * np.exp(-DELTA**2), delta_eff(DELTA), CUTOFF)
    naive = state_statespy_convention(DIM, mu, ALPHA, BETA, DELTA, CUTOFF)
    print(f"mu={mu}: |<moments|states.py mapped>| = {abs(psi.conj() @ mapped):.12f}   "
          f"naive = {abs(psi.conj() @ naive):.6f}")
    print(f"        Fock tail weight (top 5 levels) = {np.abs(psi[-5:]).max():.2e}   "
          f"nbar = {np.sum(np.arange(DIM) * abs(psi)**2):.2f}   |s| = |<0|psi>| = {abs(psi[0]):.4f}")
print(f"\nDelta_eff({DELTA}) = {delta_eff(DELTA):.6f}   (valid for Delta < {sqrt(np.log(2)/2):.4f})")

mu=0: |<moments|states.py mapped>| = 1.000000000000   naive = 0.997390
        Fock tail weight (top 5 levels) = 4.44e-03   nbar = 4.99   |s| = |<0|psi>| = 0.5768
mu=1: |<moments|states.py mapped>| = 1.000000000000   naive = 0.997414
        Fock tail weight (top 5 levels) = 3.22e-03   nbar = 5.02   |s| = |<0|psi>| = 0.2390

Delta_eff(0.3) = 0.331415   (valid for Delta < 0.5887)


### How many orders are actually needed? (the two truncations behave very differently)

This is the one place where the graded expansion can mislead, so it is worth being explicit.

* **The $n$-series converges, but slowly.** On the only column the dynamics ever sees, $q=0$, the error is
  exactly $R\mathcal N^\Delta_\mu\bigl(\sum_{p>N_{\max}}|S_p|^2/p!\bigr)^{1/2}$ — literally the Fock tail of
  $|\tilde\mu_L\rangle$, and GKP states have heavy tails. At $\Delta=0.3$ ($\bar n\approx5$) the table below
  shows $N_{\max}=16$ still leaves a $14\%$ error: $N_{\max}$ has to be comparable to the Fock cutoff
  itself, not to $\bar n$. There is no useful low-order approximation of $\sigma^x_{\rm eff}$ as an
  *operator*; the grading is a structural statement, not a small parameter.
* **The $k$-series does not converge in operator norm.** The spurious entries of §7 carry
  $\sqrt{p!q!}/(n!\,q!)\binom{q-1}{K_{\max}}$, which *grows* super-factorially with $q$. Truncating $k$
  therefore produces an operator whose spectral norm is enormous even though its action on vacuum is
  nearly exact. Since $\|H\|$ is what sets the Mandelstam–Tamm time $T=\arccos|s|/\|H\|$, quoting a speed
  limit from a $k$-truncated generator is meaningless.

The practical rule: in a Fock space of dimension $d_{\rm F}$, take $K_{\max}\ge d_{\rm F}-1$ (then $P_K$ is
*exactly* the projector on that space, §4) and use $N_{\max}$ as the only real approximation knob. The
table below separates the two effects.

In [12]:
mu = 0
psi, S_vals = state_from_moments(DIM, mu, ALPHA, BETA, DELTA, CUTOFF)
S_dict = {n: S_vals[n] for n in range(DIM)}
log_fact = np.concatenate(([0.0], np.cumsum(np.log(np.arange(1, DIM)))))
norm_N = 1.0 / np.linalg.norm(S_vals / np.exp(0.5 * log_fact))
s_ovl = norm_N * S_vals[0]
R_val = 1.0 / sqrt(1.0 - abs(s_ovl) ** 2)
print(f"N^Delta = {norm_N:.6e}   s = <0|psi> = {s_ovl:.6f}   R = {R_val:.6f}   "
      f"T_MT = arccos|s| = {np.arccos(abs(s_ovl)):.5f}")

dim_c = 26
pref = R_val * norm_N
exact_c = pref * sigma_x_exact_matrix(S_dict, dim_c)
e0 = np.zeros(dim_c); e0[0] = 1.0

print(f"\n{'Nmax':>5}{'Kmax':>6}   ||(dS)|0>||    block err (q<=Kmax)   ||sigma_trunc||_2")
for Nmax, Kmax in [(2, dim_c - 1), (6, dim_c - 1), (10, dim_c - 1), (16, dim_c - 1),
                   (dim_c - 1, dim_c - 1), (dim_c - 1, 4)]:
    T = pref * sigma_x_trunc_matrix(S_dict, Nmax, Kmax, dim_c)
    d = T - exact_c
    kk = min(Kmax, dim_c - 1) + 1
    print(f"{Nmax:>5}{Kmax:>6}   {np.linalg.norm(d @ e0):11.3e}    "
          f"{np.abs(d[:kk, :kk]).max():17.3e}    {np.linalg.norm(T, 2):14.4e}")

# same Fock window as the comparison above, so this is an equality, not an estimate
w = (np.abs(S_vals[:dim_c]) ** 2 / np.exp(log_fact[:dim_c]))
tail = np.sqrt(np.cumsum(w[::-1])[::-1])
print("\npredicted vacuum-column error pref*sqrt(sum_{Nmax<p<dim} |S_p|^2/p!):")
for Nmax in (2, 6, 10, 16):
    print(f"  Nmax={Nmax:>3}: {pref * tail[Nmax + 1]:.3e}")
print("\nLast row above is the pathology: Kmax=4 leaves the vacuum column and the q<=4 block"
      "\nessentially exact, yet the spectral norm is off by orders of magnitude.")

N^Delta = 2.862339e-01   s = <0|psi> = 0.576752+0.000000j   R = 1.224111   T_MT = arccos|s| = 0.95605

 Nmax  Kmax   ||(dS)|0>||    block err (q<=Kmax)   ||sigma_trunc||_2
    2    25     9.159e-01            6.673e-01        3.7799e-01
    6    25     7.531e-01            6.673e-01        6.4392e-01
   10    25     3.479e-01            2.403e-01        9.2776e-01
   16    25     1.408e-01            9.578e-02        9.8080e-01
   25    25     8.920e-17            1.993e-09        9.9086e-01
   25     4     8.920e-17            3.112e-16        1.3357e+06

predicted vacuum-column error pref*sqrt(sum_{Nmax<p<dim} |S_p|^2/p!):
  Nmax=  2: 9.159e-01
  Nmax=  6: 7.531e-01
  Nmax= 10: 3.479e-01
  Nmax= 16: 1.408e-01

Last row above is the pathology: Kmax=4 leaves the vacuum column and the q<=4 block
essentially exact, yet the spectral norm is off by orders of magnitude.


## 9. Where this plugs into the project

* `brachistochrone.py::quantum_brachistochrone_hamiltonian` builds the same $\sigma^{x,y}_{\rm eff}$
  numerically as dense `Qarray` outer products. The expansion here is its analytic counterpart: it says
  which ladder monomials the time-optimal generator actually contains, graded by $d=p+q$.
* For the ECD / SNAP decomposition work, the $d$-graded blocks are the natural targets: $d=1$ is a
  displacement, $d=2$ squeezing, $d\ge3$ genuinely non-Gaussian. Note again the absence of any
  number-diagonal term — SNAP phases alone cannot generate $\sigma^x_{\rm eff}$, they only enter through
  commutators with displacements.
* Sign conventions to keep straight when comparing: the notes use
  $H_{\rm opt}=\omega(\sin\phi\,\sigma^x_{\rm eff}-\cos\phi\,\sigma^y_{\rm eff})$ while `brachistochrone.py`
  uses $+\cos\phi\,\sigma^y_{\rm eff}$, and the notes' $|\psi_f^\perp\rangle$ is unnormalized (Eq. 13)
  whereas the code normalizes it. This notebook follows the code.

In [15]:
LATEX_OUT = "sigma_x_eff_expansion.tex"
Nmax, Kmax = 16, 16
body = [r"\begin{align}", r"\sigma^{x}_{\rm eff} &= R\,\mathcal N^{\Delta}_{\mu}\Bigl["]
for d, block in graded_blocks(Nmax, Kmax).items():
    body.append(rf"  &\quad + \underbrace{{{sp.latex(sp.factor_terms(block))}}}_{{d={d}}} \\")
body += [r"  &\quad + \mathcal O(d>%d)\Bigr]" % (Nmax + 2 * Kmax), r"\end{align}"]
with open(LATEX_OUT, "w") as fh:
    fh.write("\n".join(body) + "\n")
print(f"wrote {LATEX_OUT} ({Nmax=}, {Kmax=})")
print("\n".join(body[:6]) + "\n...")

wrote sigma_x_eff_expansion.tex (Nmax=16, Kmax=16)
\begin{align}
\sigma^{x}_{\rm eff} &= R\,\mathcal N^{\Delta}_{\mu}\Bigl[
  &\quad + \underbrace{\overline{{S}_{1}} {a} + {S}_{1} {{a}^\dagger}}_{d=1} \\
  &\quad + \underbrace{\frac{\overline{{S}_{2}} {a}^{2} + {S}_{2} {{a}^\dagger}^{2}}{2}}_{d=2} \\
  &\quad + \underbrace{\frac{- 6 \overline{{S}_{1}} {{a}^\dagger} {a}^{2} + \overline{{S}_{3}} {a}^{3} - 6 {S}_{1} {{a}^\dagger}^{2} {a} + {S}_{3} {{a}^\dagger}^{3}}{6}}_{d=3} \\
  &\quad + \underbrace{\frac{- 12 \overline{{S}_{2}} {{a}^\dagger} {a}^{3} + \overline{{S}_{4}} {a}^{4} - 12 {S}_{2} {{a}^\dagger}^{3} {a} + {S}_{4} {{a}^\dagger}^{4}}{24}}_{d=4} \\
...
